# Retail Intelligence — Gold Business Marts

This notebook creates business-facing Gold datasets for retail decision support.

The Retail Intelligence layer complements the customer-facing recommendation system by serving a different set of users:

- retail and category managers,
- merchandising teams,
- marketing and CRM teams,
- demand and operations teams.

The datasets are derived from the curated `workspace.cleaned_data` and existing analytical KPIs in `workspace.analysis_data`.

The first domain implemented is **Product Intelligence**.

It supports questions such as:

- Which products generate the most purchase activity?
- Which products reach the largest share of customers?
- Which products exhibit strong repeat-purchase behavior?
- Which products are important within their department?
- Which products have strong demand but weak repeat behavior?
- Which products are niche but highly loyal?

No revenue, margin, or inventory claims are made because those fields are not available in the source Instacart dataset.

In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
CLEANED_SCHEMA = "cleaned_data"
ANALYSIS_SCHEMA = "analysis_data"

print(
    f"Retail Intelligence source: "
    f"{CATALOG}.{ANALYSIS_SCHEMA}"
)

Retail Intelligence source: workspace.analysis_data


In [0]:
spark.sql(
    """
    CREATE OR REPLACE TABLE
        workspace.analysis_data.retail_product_performance
    USING DELTA
    AS

    WITH customer_base AS (

        SELECT
            COUNT(DISTINCT user_id) AS total_customers

        FROM workspace.cleaned_data.orders

        WHERE eval_set IN ('prior', 'train')
    ),

    product_base AS (

        SELECT

            p.product_id,
            p.product_name,

            p.aisle_id,
            p.aisle,

            p.department_id,
            p.department,

            p.total_purchase_count,
            p.unique_customer_count,

            p.reordered_purchase_count,
            p.first_purchase_count,

            p.product_reorder_rate,
            p.avg_add_to_cart_position,

            c.total_customers,

            SUM(
                p.total_purchase_count
            ) OVER () AS all_product_purchases,

            DENSE_RANK() OVER (
                ORDER BY
                    p.total_purchase_count DESC
            ) AS purchase_rank,

            DENSE_RANK() OVER (
                PARTITION BY p.department_id
                ORDER BY
                    p.total_purchase_count DESC
            ) AS department_purchase_rank,

            NTILE(5) OVER (
                ORDER BY
                    p.total_purchase_count DESC
            ) AS demand_quintile,

            NTILE(5) OVER (
                ORDER BY
                    p.product_reorder_rate DESC
            ) AS reorder_quintile

        FROM workspace.analysis_data.product_kpis p

        CROSS JOIN customer_base c
    )

    SELECT

        product_id,
        product_name,

        aisle_id,
        aisle,

        department_id,
        department,

        total_purchase_count,
        unique_customer_count,

        reordered_purchase_count,
        first_purchase_count,

        product_reorder_rate,
        avg_add_to_cart_position,

        ROUND(
            100.0
            * unique_customer_count
            / total_customers,
            4
        ) AS customer_penetration_pct,

        ROUND(
            100.0
            * total_purchase_count
            / all_product_purchases,
            4
        ) AS purchase_share_pct,

        ROUND(
            total_purchase_count
            / NULLIF(unique_customer_count, 0),
            2
        ) AS purchases_per_customer,

        purchase_rank,

        department_purchase_rank,

        demand_quintile,

        CASE demand_quintile
            WHEN 1 THEN 'VERY HIGH'
            WHEN 2 THEN 'HIGH'
            WHEN 3 THEN 'MEDIUM'
            WHEN 4 THEN 'LOW'
            ELSE 'VERY LOW'
        END AS demand_tier,

        reorder_quintile,

        CASE reorder_quintile
            WHEN 1 THEN 'VERY HIGH'
            WHEN 2 THEN 'HIGH'
            WHEN 3 THEN 'MEDIUM'
            WHEN 4 THEN 'LOW'
            ELSE 'VERY LOW'
        END AS repeat_behavior_tier,

        CASE
            WHEN total_purchase_count >= 100
                THEN 'HIGH'
            WHEN total_purchase_count >= 30
                THEN 'MEDIUM'
            ELSE 'LOW'
        END AS metric_reliability,

        CURRENT_TIMESTAMP()
            AS _retail_processed_at

    FROM product_base
    """
)

print(
    "retail_product_performance created successfully."
)

retail_product_performance created successfully.


In [0]:
product_performance_df = spark.table(
    "workspace.analysis_data.retail_product_performance"
)

print(
    f"Products: "
    f"{product_performance_df.count():,}"
)

display(
    product_performance_df
    .orderBy(
        F.desc("total_purchase_count")
    )
    .select(
        "product_id",
        "product_name",
        "department",
        "total_purchase_count",
        "unique_customer_count",
        "customer_penetration_pct",
        "product_reorder_rate",
        "purchases_per_customer",
        "purchase_rank",
        "department_purchase_rank",
        "demand_tier",
        "repeat_behavior_tier",
        "metric_reliability",
    )
    .limit(20)
)

Products: 49,688


product_id,product_name,department,total_purchase_count,unique_customer_count,customer_penetration_pct,product_reorder_rate,purchases_per_customer,purchase_rank,department_purchase_rank,demand_tier,repeat_behavior_tier,metric_reliability
24852,Banana,produce,491291,76125,36.9164,0.8451,6.45,1,1,VERY HIGH,VERY HIGH,HIGH
13176,Bag of Organic Bananas,produce,394930,65655,31.8391,0.8338,6.02,2,2,VERY HIGH,VERY HIGH,HIGH
21137,Organic Strawberries,produce,275577,61129,29.6442,0.7782,4.51,3,3,VERY HIGH,VERY HIGH,HIGH
21903,Organic Baby Spinach,produce,251705,56766,27.5284,0.7745,4.43,4,4,VERY HIGH,VERY HIGH,HIGH
47209,Organic Hass Avocado,produce,220877,44704,21.6790,0.7976,4.94,5,5,VERY HIGH,VERY HIGH,HIGH
47766,Organic Avocado,produce,184224,43954,21.3153,0.7614,4.19,6,6,VERY HIGH,VERY HIGH,HIGH
47626,Large Lemon,produce,160792,48614,23.5751,0.6977,3.31,7,7,VERY HIGH,VERY HIGH,HIGH
16797,Strawberries,produce,149445,44857,21.7532,0.6998,3.33,8,8,VERY HIGH,VERY HIGH,HIGH
26209,Limes,produce,146660,46658,22.6266,0.6819,3.14,9,9,VERY HIGH,VERY HIGH,HIGH
27845,Organic Whole Milk,dairy eggs,142813,24129,11.7012,0.831,5.92,10,1,VERY HIGH,VERY HIGH,HIGH


## Product Business Segmentation

Raw KPIs are useful for analysis, but business users need interpretable product groups.

Products are therefore segmented by two relative dimensions:

- **Demand** — purchase activity relative to the rest of the assortment.
- **Repeat behavior** — tendency for purchases to be reorders.

Metric reliability is also considered to avoid over-interpreting products with very little purchase history.

In [0]:
spark.sql(
    """
    CREATE OR REPLACE TABLE
        workspace.analysis_data.retail_product_segments
    USING DELTA
    AS

    SELECT
        *,

        CASE

            WHEN metric_reliability = 'LOW'
                THEN 'INSUFFICIENT HISTORY'

            WHEN demand_quintile <= 2
                 AND reorder_quintile <= 2
                THEN 'CORE STAPLE'

            WHEN demand_quintile <= 2
                 AND reorder_quintile >= 4
                THEN 'HIGH REACH / LOW REPEAT'

            WHEN demand_quintile >= 4
                 AND reorder_quintile <= 2
                THEN 'NICHE LOYAL'

            WHEN demand_quintile >= 4
                 AND reorder_quintile >= 4
                THEN 'LOW TRACTION'

            ELSE 'BALANCED'

        END AS product_segment,

        CASE

            WHEN metric_reliability = 'LOW'
                THEN 'Collect more history'

            WHEN demand_quintile <= 2
                 AND reorder_quintile <= 2
                THEN 'Protect core demand'

            WHEN demand_quintile <= 2
                 AND reorder_quintile >= 4
                THEN 'Investigate repeat-purchase opportunity'

            WHEN demand_quintile >= 4
                 AND reorder_quintile <= 2
                THEN 'Nurture loyal niche demand'

            WHEN demand_quintile >= 4
                 AND reorder_quintile >= 4
                THEN 'Monitor product engagement'

            ELSE 'Monitor performance'

        END AS business_focus

    FROM workspace.analysis_data.retail_product_performance
    """
)

print(
    "retail_product_segments created successfully."
)

retail_product_segments created successfully.


In [0]:
product_performance_df = spark.table(
    "workspace.analysis_data.retail_product_performance"
)

print(
    f"Products: "
    f"{product_performance_df.count():,}"
)

display(
    product_performance_df
    .orderBy(
        F.desc("total_purchase_count")
    )
    .select(
        "product_id",
        "product_name",
        "department",
        "total_purchase_count",
        "unique_customer_count",
        "customer_penetration_pct",
        "product_reorder_rate",
        "purchases_per_customer",
        "purchase_rank",
        "department_purchase_rank",
        "demand_tier",
        "repeat_behavior_tier",
        "metric_reliability",
    )
    .limit(20)
)

Products: 49,688


product_id,product_name,department,total_purchase_count,unique_customer_count,customer_penetration_pct,product_reorder_rate,purchases_per_customer,purchase_rank,department_purchase_rank,demand_tier,repeat_behavior_tier,metric_reliability
24852,Banana,produce,491291,76125,36.9164,0.8451,6.45,1,1,VERY HIGH,VERY HIGH,HIGH
13176,Bag of Organic Bananas,produce,394930,65655,31.8391,0.8338,6.02,2,2,VERY HIGH,VERY HIGH,HIGH
21137,Organic Strawberries,produce,275577,61129,29.6442,0.7782,4.51,3,3,VERY HIGH,VERY HIGH,HIGH
21903,Organic Baby Spinach,produce,251705,56766,27.5284,0.7745,4.43,4,4,VERY HIGH,VERY HIGH,HIGH
47209,Organic Hass Avocado,produce,220877,44704,21.6790,0.7976,4.94,5,5,VERY HIGH,VERY HIGH,HIGH
47766,Organic Avocado,produce,184224,43954,21.3153,0.7614,4.19,6,6,VERY HIGH,VERY HIGH,HIGH
47626,Large Lemon,produce,160792,48614,23.5751,0.6977,3.31,7,7,VERY HIGH,VERY HIGH,HIGH
16797,Strawberries,produce,149445,44857,21.7532,0.6998,3.33,8,8,VERY HIGH,VERY HIGH,HIGH
26209,Limes,produce,146660,46658,22.6266,0.6819,3.14,9,9,VERY HIGH,VERY HIGH,HIGH
27845,Organic Whole Milk,dairy eggs,142813,24129,11.7012,0.831,5.92,10,1,VERY HIGH,VERY HIGH,HIGH


## Product Business Segmentation

Raw KPIs are useful for analysis, but business users need interpretable product groups.

Products are therefore segmented by two relative dimensions:

- **Demand** — purchase activity relative to the rest of the assortment.
- **Repeat behavior** — tendency for purchases to be reorders.

Metric reliability is also considered to avoid over-interpreting products with very little purchase history.

In [0]:
spark.sql(
    """
    CREATE OR REPLACE TABLE
        workspace.analysis_data.retail_product_segments
    USING DELTA
    AS

    SELECT
        *,

        CASE

            WHEN metric_reliability = 'LOW'
                THEN 'INSUFFICIENT HISTORY'

            WHEN demand_quintile = 1
                 AND reorder_quintile = 1
                THEN 'CORE STAPLE'

            WHEN demand_quintile = 1
                 AND reorder_quintile >= 4
                THEN 'HIGH REACH / LOW REPEAT'

            WHEN demand_quintile >= 4
                 AND reorder_quintile = 1
                THEN 'NICHE LOYAL'

            WHEN demand_quintile >= 4
                 AND reorder_quintile >= 4
                THEN 'LOW TRACTION'

            ELSE 'BALANCED'

        END AS product_segment,

        CASE

            WHEN metric_reliability = 'LOW'
                THEN 'Collect more history'

            WHEN demand_quintile = 1
                 AND reorder_quintile = 1
                THEN 'Protect core demand'

            WHEN demand_quintile = 1
                 AND reorder_quintile >= 4
                THEN 'Investigate repeat-purchase opportunity'

            WHEN demand_quintile >= 4
                 AND reorder_quintile = 1
                THEN 'Nurture loyal niche demand'

            WHEN demand_quintile >= 4
                 AND reorder_quintile >= 4
                THEN 'Monitor product engagement'

            ELSE 'Monitor performance'

        END AS business_focus

    FROM workspace.analysis_data.retail_product_performance
    """
)

print(
    "retail_product_segments created successfully."
)

retail_product_segments created successfully.


In [0]:
segments_df = spark.table(
    "workspace.analysis_data.retail_product_segments"
)

display(
    segments_df
    .groupBy(
        "product_segment"
    )
    .agg(
        F.count("*").alias(
            "product_count"
        ),

        F.round(
            F.avg(
                "total_purchase_count"
            ),
            2,
        ).alias(
            "avg_purchases"
        ),

        F.round(
            F.avg(
                "product_reorder_rate"
            )
            * 100,
            2,
        ).alias(
            "avg_reorder_rate_pct"
        ),

        F.round(
            F.avg(
                "customer_penetration_pct"
            ),
            4,
        ).alias(
            "avg_customer_penetration_pct"
        ),
    )
    .orderBy(
        F.desc("avg_purchases")
    )
)

product_segment,product_count,avg_purchases,avg_reorder_rate_pct,avg_customer_penetration_pct
CORE STAPLE,4343,4810.01,64.83,0.7433
HIGH REACH / LOW REPEAT,930,1167.76,22.87,0.4361
BALANCED,25508,453.18,42.51,0.1172
LOW TRACTION,1212,33.59,18.23,0.0133
NICHE LOYAL,330,33.54,66.15,0.0055
INSUFFICIENT HISTORY,17365,13.35,23.0,0.0047


In [0]:
display(
    segments_df
    .filter(
        F.col("metric_reliability")
        == "HIGH"
    )
    .select(
        "product_name",
        "department",
        "total_purchase_count",
        "customer_penetration_pct",
        "product_reorder_rate",
        "demand_tier",
        "repeat_behavior_tier",
        "product_segment",
        "business_focus",
    )
    .orderBy(
        F.desc("total_purchase_count")
    )
    .limit(30)
)

product_name,department,total_purchase_count,customer_penetration_pct,product_reorder_rate,demand_tier,repeat_behavior_tier,product_segment,business_focus
Banana,produce,491291,36.9164,0.8451,VERY HIGH,VERY HIGH,CORE STAPLE,Protect core demand
Bag of Organic Bananas,produce,394930,31.8391,0.8338,VERY HIGH,VERY HIGH,CORE STAPLE,Protect core demand
Organic Strawberries,produce,275577,29.6442,0.7782,VERY HIGH,VERY HIGH,CORE STAPLE,Protect core demand
Organic Baby Spinach,produce,251705,27.5284,0.7745,VERY HIGH,VERY HIGH,CORE STAPLE,Protect core demand
Organic Hass Avocado,produce,220877,21.6790,0.7976,VERY HIGH,VERY HIGH,CORE STAPLE,Protect core demand
Organic Avocado,produce,184224,21.3153,0.7614,VERY HIGH,VERY HIGH,CORE STAPLE,Protect core demand
Large Lemon,produce,160792,23.5751,0.6977,VERY HIGH,VERY HIGH,CORE STAPLE,Protect core demand
Strawberries,produce,149445,21.7532,0.6998,VERY HIGH,VERY HIGH,CORE STAPLE,Protect core demand
Limes,produce,146660,22.6266,0.6819,VERY HIGH,VERY HIGH,CORE STAPLE,Protect core demand
Organic Whole Milk,dairy eggs,142813,11.7012,0.831,VERY HIGH,VERY HIGH,CORE STAPLE,Protect core demand


## Gold Data Quality Validation

Before the Retail Intelligence datasets are used by dashboards or downstream applications,
the Gold outputs are validated for:

- product uniqueness,
- source-to-target row preservation,
- required business fields,
- valid percentage/rate ranges,
- non-negative purchase metrics,
- successful business segmentation.

These checks protect downstream BI consumers from malformed or incomplete analytical data.

In [0]:
from pyspark.sql import functions as F

segments_df = spark.table(
    "workspace.analysis_data.retail_product_segments"
)

quality_checks_df = (
    segments_df
    .agg(

        F.count("*").alias(
            "row_count"
        ),

        F.countDistinct(
            "product_id"
        ).alias(
            "distinct_product_count"
        ),

        F.sum(
            F.when(
                F.col("product_id").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "missing_product_id"
        ),

        F.sum(
            F.when(
                F.col("product_name").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "missing_product_name"
        ),

        F.sum(
            F.when(
                F.col("department").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "missing_department"
        ),

        F.sum(
            F.when(
                (
                    F.col("product_reorder_rate") < 0
                )
                |
                (
                    F.col("product_reorder_rate") > 1
                ),
                1
            ).otherwise(0)
        ).alias(
            "invalid_reorder_rate"
        ),

        F.sum(
            F.when(
                (
                    F.col("customer_penetration_pct") < 0
                )
                |
                (
                    F.col("customer_penetration_pct") > 100
                ),
                1
            ).otherwise(0)
        ).alias(
            "invalid_customer_penetration"
        ),

        F.sum(
            F.when(
                F.col("total_purchase_count") < 0,
                1
            ).otherwise(0)
        ).alias(
            "negative_purchase_count"
        ),

        F.sum(
            F.when(
                F.col("product_segment").isNull(),
                1
            ).otherwise(0)
        ).alias(
            "missing_product_segment"
        ),
    )
)

display(quality_checks_df)

row_count,distinct_product_count,missing_product_id,missing_product_name,missing_department,invalid_reorder_rate,invalid_customer_penetration,negative_purchase_count,missing_product_segment
49688,49688,0,0,0,0,0,0,0


In [0]:
source_product_count = (
    spark.table(
        "workspace.analysis_data.product_kpis"
    )
    .select("product_id")
    .distinct()
    .count()
)

gold_product_count = (
    segments_df
    .select("product_id")
    .distinct()
    .count()
)

print(
    f"Source products: {source_product_count:,}"
)

print(
    f"Gold products:   {gold_product_count:,}"
)

print(
    f"Difference:      "
    f"{gold_product_count - source_product_count:,}"
)

Source products: 49,688
Gold products:   49,688
Difference:      0


In [0]:
quality = (
    quality_checks_df
    .first()
    .asDict()
)

assert (
    quality["row_count"]
    == quality["distinct_product_count"]
), "Duplicate product_id values detected."

assert (
    quality["missing_product_id"] == 0
), "Missing product IDs detected."

assert (
    quality["missing_product_name"] == 0
), "Missing product names detected."

assert (
    quality["missing_department"] == 0
), "Missing departments detected."

assert (
    quality["invalid_reorder_rate"] == 0
), "Invalid reorder rates detected."

assert (
    quality["invalid_customer_penetration"] == 0
), "Invalid customer penetration values detected."

assert (
    quality["negative_purchase_count"] == 0
), "Negative purchase counts detected."

assert (
    quality["missing_product_segment"] == 0
), "Products without business segments detected."

assert (
    source_product_count
    == gold_product_count
), "Source-to-Gold product count mismatch."

print(
    "✓ Retail Product Intelligence Gold quality checks passed."
)

✓ Retail Product Intelligence Gold quality checks passed.
